In [1]:
import re
import pandas as pd
from typing import List, Dict, Optional


def parse_hyperparameters(log_chunk: str) -> Dict:
    """Extracts hyperparameters from a log chunk for a single run."""

    model_params_pattern = re.compile(
        r"Model Params:\n"
        r"\s+LR=([\d.]+)\n"
        r"\s+INNER_DIM=(\d+)\n"
        r"\s+BATCH_SIZE=(\d+)\n"
        r"\s+SOFTMAX_ASSIGN=(\w+)\n"
        r"\s+DECREASE_PROPORTION=([\d.]+)\s+Repetition=(\d+)",
        re.DOTALL
    )

    loss_config_pattern = re.compile(
        r"Loss Config \((\w+)\):\n"
        r".*?LINK\s+=\s+([\d.]+)\n"
        r".*?ENTROPY\s+=\s+([\d.]+)\n"
        r".*?RECONSTRUCTION\s+=\s+([\d.]+)\n"
        r".*?CONTRASTIVE\s+=\s+([\d.]+)\n"
        r".*?BALANCE\s+=\s+([\d.]+)\n"
        r".*?REPEL\s+=\s+([\d.]+)\n"
        r".*?L2\s+=\s+([\d.]+)",
        re.DOTALL
    )

    run_id_pattern = re.search(r"Starting Run (\d+/\d+)", log_chunk)
    loss_config_id_match = re.search(r"Loss Config ID: (cfg_\d+)", log_chunk)
    model_match = model_params_pattern.search(log_chunk)
    loss_match = loss_config_pattern.search(log_chunk)

    params = {}
    params['Run'] = run_id_pattern.group(1) if run_id_pattern else None
    params['Loss_Config_ID'] = loss_config_id_match.group(1) if loss_config_id_match else None

    if model_match:
        params['LR'] = float(model_match.group(1))
        params['INNER_DIM'] = int(model_match.group(2))
        params['BATCH_SIZE'] = int(model_match.group(3))
        params['SOFTMAX_ASSIGN'] = model_match.group(4) == 'True'
        params['DECREASE_PROPORTION'] = float(model_match.group(5))
        params['Repetition'] = int(model_match.group(6))

    if loss_match:
        params['LINK'] = float(loss_match.group(2))
        params['ENTROPY'] = float(loss_match.group(3))
        params['RECONSTRUCTION'] = float(loss_match.group(4))
        params['CONTRASTIVE'] = float(loss_match.group(5))
        params['BALANCE'] = float(loss_match.group(6))
        params['REPEL'] = float(loss_match.group(7))
        params['L2'] = float(loss_match.group(8))

    return params


def parse_results(log_chunk: str) -> Optional[Dict]:
    """Extracts final results, metrics from the best epoch, and the best model path."""

    result_summary_match = re.search(
        r"\[Result\] ID: .* \| Best E-Score: ([\d.]+) at epoch (\d+)",
        log_chunk
    )
    if not result_summary_match:
        return None

    best_escore = float(result_summary_match.group(1))
    best_epoch = int(result_summary_match.group(2))

    epoch_pattern = re.compile(
        rf"Epoch\s+{best_epoch:03d}\s+\| Acc: ([\d.]+)\s+\| F1: ([\d.]+)\s+\|"
        rf" Comp: ([\d.]+)\s+\| Conf: ([\d.]+)\s+\| Mod: ([\d.]+)\s+\|"
        rf" Sil: ([\d.]+)\s+\| E-Score: [\d.]+\s+\| Sum Params: ([\d.]+)"
    )

    epoch_match = epoch_pattern.search(log_chunk)
    if not epoch_match:
        return None

    num_params_match = re.search(r"Model Initialized with (\d+) parameters", log_chunk)
    num_params = int(num_params_match.group(1)) if num_params_match else 0

    sum_params = float(epoch_match.group(7))
    avg_param_value = sum_params / num_params if num_params > 0 else 0

    # --- NEW ---: Extract the best model path
    model_path_pattern = re.compile(r"✅ New best model saved with E-Score: [\d.]+ into '(.+?)'")
    saved_paths = model_path_pattern.findall(log_chunk)
    best_model_path = saved_paths[-1] if saved_paths else None
    # --- END NEW ---

    results = {
        'Best_Epoch': best_epoch,
        'Num_Parameters': num_params,
        'Acc': float(epoch_match.group(1)),
        'F1': float(epoch_match.group(2)),
        'Comp': float(epoch_match.group(3)),
        'Conf': float(epoch_match.group(4)),
        'Mod': float(epoch_match.group(5)),
        'Sil': float(epoch_match.group(6)),
        'Best_E-Score': best_escore,
        'Sum_Params_at_Best': sum_params,
        'Avg_Param_Value_at_Best': avg_param_value,
        'Best_Model_Path': best_model_path  # --- NEW ---
    }

    return results


def parse_log_file(log_file_path: str) -> List[Dict]:
    """Parses the entire log file and returns a list of dictionaries with results."""

    with open(log_file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    runs = re.split(
        r"====================================================================================================",
        content)

    all_data = []
    for run_chunk in runs:
        if "Starting Run" not in run_chunk:
            continue

        hyperparams = parse_hyperparameters(run_chunk)
        results = parse_results(run_chunk)

        if hyperparams and results:
            combined_data = {**hyperparams, **results}
            all_data.append(combined_data)

    return all_data


In [2]:

log_file = '../logs/training_model_experiment_gridsearch_STF_HC_Voto_Relatorio_loss_20250903_135039.log'
output_excel_file = 'experiment_results_mix.xlsx'

experiment_data = parse_log_file(log_file)

if experiment_data:
    df = pd.DataFrame(experiment_data)

    hyperparam_cols = [
        'Run', 'Loss_Config_ID', 'LR', 'INNER_DIM', 'BATCH_SIZE',
        'SOFTMAX_ASSIGN', 'DECREASE_PROPORTION', 'Repetition', 'LINK', 'ENTROPY',
        'RECONSTRUCTION', 'CONTRASTIVE', 'BALANCE', 'REPEL', 'L2'
    ]
    result_cols = [
        'Best_Epoch', 'Acc', 'F1', 'Comp', 'Conf', 'Mod', 'Sil', 'Best_E-Score',
        'Num_Parameters', 'Sum_Params_at_Best', 'Avg_Param_Value_at_Best',
        'Best_Model_Path'  # --- NEW ---
    ]

    final_cols = [col for col in hyperparam_cols + result_cols if col in df.columns]
    df = df[final_cols]

    df.to_excel(output_excel_file, index=False)
    print(f"✅ Successfully parsed {len(df)} runs.")
    print(f"📄 Results saved to '{output_excel_file}'")
else:
    print("❌ No valid experiment runs found in the log file.")

✅ Successfully parsed 57 runs.
📄 Results saved to 'experiment_results_mix.xlsx'


In [3]:
log_file = '../logs/training_model_experiment_gridsearch_STF_HC_Voto_Relatorio_loss.log'
output_excel_file = 'experiment_results_isolated.xlsx'

experiment_data = parse_log_file(log_file)

if experiment_data:
    df = pd.DataFrame(experiment_data)

    hyperparam_cols = [
        'Run', 'Loss_Config_ID', 'LR', 'INNER_DIM', 'BATCH_SIZE',
        'SOFTMAX_ASSIGN', 'DECREASE_PROPORTION', 'Repetition', 'LINK', 'ENTROPY',
        'RECONSTRUCTION', 'CONTRASTIVE', 'BALANCE', 'REPEL', 'L2'
    ]
    result_cols = [
        'Best_Epoch', 'Acc', 'F1', 'Comp', 'Conf', 'Mod', 'Sil', 'Best_E-Score',
        'Num_Parameters', 'Sum_Params_at_Best', 'Avg_Param_Value_at_Best',
        'Best_Model_Path'  # --- NEW ---
    ]

    final_cols = [col for col in hyperparam_cols + result_cols if col in df.columns]
    df = df[final_cols]

    df.to_excel(output_excel_file, index=False)
    print(f"✅ Successfully parsed {len(df)} runs.")
    print(f"📄 Results saved to '{output_excel_file}'")
else:
    print("❌ No valid experiment runs found in the log file.")


✅ Successfully parsed 216 runs.
📄 Results saved to 'experiment_results_isolated.xlsx'


In [4]:
# Read first dataset
df1 = pd.read_excel("experiment_results_isolated.xlsx")

# Apply filtering (need parentheses for each condition)
df1 = df1[(df1["INNER_DIM"] == 32) & (df1["DECREASE_PROPORTION"] == 0.1)]

# Read second dataset
df2 = pd.read_excel("experiment_results_mix.xlsx")

# Concatenate both
df = pd.concat([df1, df2], ignore_index=True)

# Save merged results
df.to_excel("experiment_results_full.xlsx", index=False)

In [5]:
# 3. Calculate median and standard deviation for each hyperparameter group
# We use .agg() to apply multiple summary functions at once.

hyperparam_cols = [
    'Loss_Config_ID', 'LR', 'INNER_DIM', 'BATCH_SIZE', 'SOFTMAX_ASSIGN',
    'DECREASE_PROPORTION', 'LINK', 'ENTROPY', 'RECONSTRUCTION',
    'CONTRASTIVE', 'BALANCE', 'REPEL', 'L2'
]
result_cols = [
    'Acc', 'F1', 'Comp', 'Conf', 'Mod', 'Sil', 'Best_E-Score',
    'Avg_Param_Value_at_Best'
]


# 3. Calculate median and standard deviation for each hyperparameter group
# We use .agg() to apply multiple summary functions at once.
agg_functions = {col: ['median', 'std'] for col in result_cols}
stats_df = df.groupby(hyperparam_cols).agg(agg_functions).reset_index()

# Clean up the multi-level column names (e.g., ('Acc', 'median') -> 'Acc_median')
stats_df.columns = ['_'.join(col).strip('_') for col in stats_df.columns.values]

# 4. Find the model path corresponding to the median 'Best_E-Score' for each group
# This is a multi-step process:
# a. For each row, calculate its group's median 'Best_E-Score'
df['median_e_score_for_group'] = df.groupby(hyperparam_cols)['Best_E-Score'].transform('median')

# b. Find the absolute difference between the row's score and its group's median
df['dist_to_median'] = (df['Best_E-Score'] - df['median_e_score_for_group']).abs()

# c. For each group, find the index of the row with the smallest difference (i.e., the true median run)
# .loc allows us to select these specific rows from the original dataframe
idx_of_median_runs = df.groupby(hyperparam_cols)['dist_to_median'].idxmin()
median_run_info = df.loc[idx_of_median_runs][hyperparam_cols + ['Best_Model_Path']]
median_run_info = median_run_info.rename(columns={"Best_Model_Path": "Median_E-Score_Model_Path"})

# 5. Merge the statistics with the median model path
final_report = pd.merge(stats_df, median_run_info, on=hyperparam_cols)
final_report = final_report.round(3)

# 6. Display the final report
# Set display options for better viewing
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print("--- Aggregated Hyperparameter Performance Report ---")
final_report

--- Aggregated Hyperparameter Performance Report ---


,Loss_Config_ID,LR,INNER_DIM,BATCH_SIZE,SOFTMAX_ASSIGN,DECREASE_PROPORTION,LINK,ENTROPY,RECONSTRUCTION,CONTRASTIVE,BALANCE,REPEL,L2,Acc_median,Acc_std,F1_median,F1_std,Comp_median,Comp_std,Conf_median,Conf_std,Mod_median,Mod_std,Sil_median,Sil_std,Best_E-Score_median,Best_E-Score_std,Avg_Param_Value_at_Best_median,Avg_Param_Value_at_Best_std,Median_E-Score_Model_Path
0,cfg_00000,0.0,32,4,True,0.1,0,0.00,0.00,0.00,0.00,0.00,0.00,0.851,0.016,0.803,0.028,0.995,0.003,0.874,0.006,0.341,0.001,0.392,0.000,0.864,0.017,0.076,0.000,NaN
1,cfg_00000,0.0,32,4,True,0.1,0,0.00,0.00,0.00,0.00,0.00,0.01,0.856,0.026,0.806,0.021,1.000,0.009,0.874,0.003,0.337,0.001,0.389,0.002,0.866,0.014,0.080,0.000,models/grid_search/STF_HC_Voto_Relatorio/best_...
2,cfg_00001,0.0,32,4,True,0.1,1,0.00,0.00,0.00,0.00,0.00,0.00,0.831,0.021,0.787,0.031,1.000,0.000,0.869,0.002,0.339,0.001,0.390,0.002,0.854,0.018,0.076,0.000,NaN
3,cfg_00002,0.0,32,4,True,0.1,10,0.00,0.00,0.00,0.00,0.00,0.00,0.841,0.030,0.790,0.034,0.995,0.005,0.874,0.006,0.337,0.002,0.390,0.002,0.857,0.021,0.076,0.000,NaN
4,cfg_00003,0.0,32,4,True,0.1,100,0.00,0.00,0.00,0.00,0.00,0.00,0.826,0.019,0.782,0.029,1.000,0.000,0.873,0.001,0.340,0.002,0.391,0.001,0.852,0.017,0.076,0.000,NaN
5,cfg_00004,0.0,32,4,True,0.1,1000,0.00,0.00,0.00,0.00,0.00,0.00,0.836,0.022,0.776,0.023,1.000,0.006,0.880,0.005,0.337,0.003,0.392,0.002,0.849,0.013,0.076,0.000,NaN
6,cfg_00005,0.0,32,4,True,0.1,0,0.01,0.00,0.00,0.00,0.00,0.00,0.856,0.005,0.803,0.001,1.000,0.000,0.866,0.007,0.344,0.000,0.397,0.000,0.864,0.002,0.077,0.000,NaN
7,cfg_00006,0.0,32,4,True,0.1,0,0.10,0.00,0.00,0.00,0.00,0.00,0.851,0.006,0.811,0.004,1.000,0.020,0.868,0.002,0.343,0.002,0.403,0.002,0.868,0.007,0.077,0.000,NaN
8,cfg_00007,0.0,32,4,True,0.1,0,1.00,0.00,0.00,0.00,0.00,0.00,0.841,0.010,0.790,0.026,1.000,0.009,0.847,0.007,0.346,0.003,0.407,0.001,0.850,0.013,0.077,0.000,NaN
9,cfg_00008,0.0,32,4,True,0.1,0,10.00,0.00,0.00,0.00,0.00,0.00,0.826,0.015,0.782,0.006,1.000,0.003,0.827,0.002,0.345,0.004,0.408,0.004,0.842,0.003,0.076,0.000,NaN


In [6]:
print(final_report)

   Loss_Config_ID   LR  INNER_DIM  BATCH_SIZE  SOFTMAX_ASSIGN  DECREASE_PROPORTION  LINK  ENTROPY  RECONSTRUCTION  CONTRASTIVE  BALANCE  REPEL     L2  Acc_median  Acc_std  F1_median  F1_std  \
0       cfg_00000  0.0         32           4            True                  0.1     0     0.00            0.00         0.00     0.00   0.00   0.00       0.851    0.016      0.803   0.028   
1       cfg_00000  0.0         32           4            True                  0.1     0     0.00            0.00         0.00     0.00   0.00   0.01       0.856    0.026      0.806   0.021   
2       cfg_00001  0.0         32           4            True                  0.1     1     0.00            0.00         0.00     0.00   0.00   0.00       0.831    0.021      0.787   0.031   
3       cfg_00002  0.0         32           4            True                  0.1    10     0.00            0.00         0.00     0.00   0.00   0.00       0.841    0.030      0.790   0.034   
4       cfg_00003  0.0         32  

In [7]:
final_report.to_excel("experiments_results_full_med_std.xlsx", index=False)

In [8]:
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings

warnings.filterwarnings("ignore")

# Load the parsed data
try:
    df = pd.read_excel("experiments_results_full_med_std.xlsx")
except FileNotFoundError:
    print("Error: 'experiment_results_full.xlsx' not found.")
    exit()

# Define independent (loss terms) and dependent (metrics) variables
loss_terms = ['LINK', 'ENTROPY', 'RECONSTRUCTION', 'CONTRASTIVE', 'BALANCE', 'REPEL', 'L2']
metrics = ['Acc_median', 'F1_median', 'Comp_median', 'Conf_median', 'Mod_median',
           'Sil_median', 'Best_E-Score_median']

# Ensure numeric conversion
for col in loss_terms + metrics:
    if col not in df.columns:
        print(f"Warning: Column '{col}' not found in the DataFrame. Skipping.")
        if col in loss_terms: loss_terms.remove(col)
        if col in metrics: metrics.remove(col)

df[loss_terms + metrics] = df[loss_terms + metrics].apply(pd.to_numeric, errors='coerce')
df.dropna(subset=loss_terms + metrics, inplace=True)

print("=" * 60)
print("🔎 Method 1: ANOVA / Kruskal-Wallis Significance Tests")
print("=" * 60)

anova_results = []
for metric in metrics:
    for term in loss_terms:
        groups = [df.loc[df[term] == val, metric] for val in df[term].unique()]
        # Only test if there are at least 2 groups with >1 sample
        valid_groups = [g for g in groups if len(g) > 1]
        if len(valid_groups) >= 2:
            stat, p = stats.kruskal(*valid_groups)
            anova_results.append({
                "Metric": metric,
                "Loss Term": term,
                "Test": "Kruskal-Wallis",
                "Statistic": stat,
                "P-Value": p
            })

anova_df = pd.DataFrame(anova_results)
print(anova_df.sort_values("P-Value").to_string())


print("\n\n" + "=" * 60)
print("🔬 Method 2: Multiple Linear Regression with VIF")
print("=" * 60)

for metric in metrics:
    print(f"\n--- Analyzing Metric: {metric} ---")

    # OLS regression
    X = df[loss_terms]
    y = df[metric]
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()
    print(model.summary())

    # Variance Inflation Factor (multicollinearity check)
    vif_data = pd.DataFrame()
    vif_data["Feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    print("\nVIF (Variance Inflation Factor):")
    print(vif_data)

    print("\n" + "-" * 80)

print("\n\n" + "=" * 60)
print("🧪 Method 3: Mixed-Effects Regression (Optional)")
print("=" * 60)
print("Treats each config as fixed effect and 'seed/run' (if available) as random effect.\n")

# Example: if 'Run_ID' column exists for seeds
if "Run_ID" in df.columns:
    for metric in metrics:
        print(f"\n--- Mixed Effects for Metric: {metric} ---")
        formula = f"{metric} ~ " + " + ".join(loss_terms)
        try:
            model = smf.mixedlm(formula, df, groups=df["Run_ID"])
            result = model.fit()
            print(result.summary())
        except Exception as e:
            print(f"Mixed model failed for {metric}: {e}")

🔎 Method 1: ANOVA / Kruskal-Wallis Significance Tests
                 Metric       Loss Term            Test  Statistic   P-Value
25          Conf_median         BALANCE  Kruskal-Wallis  20.386727  0.000141
32           Mod_median         BALANCE  Kruskal-Wallis  18.077261  0.000424
46  Best_E-Score_median         BALANCE  Kruskal-Wallis  15.979734  0.001145
33           Mod_median           REPEL  Kruskal-Wallis   9.644496  0.021843
40           Sil_median           REPEL  Kruskal-Wallis   9.503409  0.023295
11            F1_median         BALANCE  Kruskal-Wallis   9.101993  0.027965
44  Best_E-Score_median  RECONSTRUCTION  Kruskal-Wallis   8.994308  0.029367
23          Conf_median  RECONSTRUCTION  Kruskal-Wallis   8.606014  0.035015
2            Acc_median  RECONSTRUCTION  Kruskal-Wallis   8.423117  0.038031
4            Acc_median         BALANCE  Kruskal-Wallis   7.736288  0.051788
9             F1_median  RECONSTRUCTION  Kruskal-Wallis   7.700424  0.052626
27          Conf_media

In [9]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler


def analyze_and_create_table(df: pd.DataFrame, loss_terms: list, metrics: list) -> pd.DataFrame:
    """
    Performs multiple linear regression for each metric and returns a summary table.
    """
    # Initialize a DataFrame to store the formatted results
    results_df = pd.DataFrame(index=loss_terms + ['Adjusted R²'], columns=metrics)

    for metric in metrics:
        # Prepare data, removing any rows with missing values for this specific metric
        temp_df = df[loss_terms + [metric]].dropna()
        X_raw = temp_df[loss_terms]
        y_raw = temp_df[[metric]]

        # Standardize both independent and dependent variables
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        X_scaled = scaler_X.fit_transform(X_raw)
        y_scaled = scaler_y.fit_transform(y_raw).flatten()

        X_scaled_df = pd.DataFrame(X_scaled, columns=loss_terms)
        X_with_const = sm.add_constant(X_scaled_df)

        # Fit the OLS model
        model = sm.OLS(y_scaled, X_with_const).fit()

        # Store the Adjusted R-squared value
        results_df.loc['Adjusted R²', metric] = f"{model.rsquared_adj:.2f}"

        # Extract coefficients and p-values
        for term in loss_terms:
            coef = model.params[term]
            p_value = model.pvalues[term]

            # Determine significance asterisks
            if p_value < 0.001:
                stars = '***'
            elif p_value < 0.01:
                stars = '**'
            elif p_value < 0.05:
                stars = '*'
            else:
                stars = ''

            # Format the cell entry
            sign = '+' if coef > 0 else ''
            results_df.loc[term, metric] = f"{sign}{coef:.2f}{stars}"

    return results_df


# --- Main Execution ---
if __name__ == "__main__":
    try:
        df = pd.read_excel('experiment_results_full.xlsx')
    except FileNotFoundError:
        print("Error: 'experiment_results.xlsx' not found.")
        print("Please run the previous parsing script first.")
        exit()

    loss_terms = ['LINK', 'ENTROPY', 'RECONSTRUCTION', 'CONTRASTIVE', 'BALANCE', 'REPEL', 'L2']
    metrics = ['Acc', 'F1', 'Comp', 'Conf', 'Mod', 'Sil', 'Best_E-Score']

    # Generate the publication-ready table
    final_table = analyze_and_create_table(df, loss_terms, metrics)

    print("### Publication-Ready Results Table (Markdown Format) ###\n")
    print(final_table.to_markdown())

### Publication-Ready Results Table (Markdown Format) ###

|                |   Acc |    F1 | Comp   | Conf     | Mod      | Sil      | Best_E-Score   |
|:---------------|------:|------:|:-------|:---------|:---------|:---------|:---------------|
| LINK           | -0.05 | -0.08 | +0.01  | +0.07    | -0.04    | -0.11    | -0.04          |
| ENTROPY        | -0.12 | -0.05 | +0.04  | -0.18*   | +0.12    | +0.16*   | -0.11          |
| RECONSTRUCTION | -0.12 | -0.08 | -0.00  | +0.12    | -0.15*   | -0.16*   | -0.02          |
| CONTRASTIVE    | -0.11 | -0.08 | -0.05  | -0.31*** | +0.09    | +0.08    | -0.20*         |
| BALANCE        |  0.09 |  0.1  | -0.05  | +0.16*   | -0.17*   | -0.01    | +0.14          |
| REPEL          | -0.13 | -0.14 | +0.02  | -0.37*** | +0.41*** | +0.48*** | -0.27**        |
| L2             | -0.02 | -0.04 | -0.21* | +0.02    | -0.03    | -0.08    | -0.05          |
| Adjusted R²    |  0.01 | -0    | 0.00   | 0.27     | 0.22     | 0.28     | 0.10           |


In [10]:
import pandas as pd

def find_best_combination(df: pd.DataFrame, active_terms: list, search_type: str = 'strict'):
    """
    Finds the best-performing run for a specific combination of active loss terms.

    Args:
        df: DataFrame with experiment results.
        active_terms: A list of loss term names that must be active (> 0).
        search_type: 'strict' (only active_terms are non-zero) or
                     'relaxed' (active_terms must be non-zero, others can be anything).
    """

    # Start with a copy of the full dataframe
    filtered_df = df.copy()

    # --- Build the filter based on the search criteria ---
    # Condition 1: All specified active terms must have lambdas > 0
    for term in active_terms:
        if term in filtered_df.columns:
            filtered_df = filtered_df[filtered_df[term] > 0]

    # Condition 2 (for 'strict' search only): All other terms must be 0
    if search_type == 'strict':
        all_loss_terms = ['LINK', 'ENTROPY', 'RECONSTRUCTION', 'CONTRASTIVE', 'BALANCE', 'REPEL', 'L2']
        inactive_terms = [t for t in all_loss_terms if t not in active_terms]
        for term in inactive_terms:
            if term in filtered_df.columns:
                filtered_df = filtered_df[filtered_df[term] == 0]

    # Sort by the primary metric to find the best run
    filtered_df.sort_values(by='Best_E-Score', ascending=False, inplace=True)

    if not filtered_df.empty:
        return filtered_df.iloc[0]
    else:
        return None

# --- Main Execution ---
if __name__ == "__main__":
    results_file = 'experiment_results_full.xlsx'
    try:
        df = pd.read_excel(results_file)
    except FileNotFoundError:
        print(f"Error: '{results_file}' not found. Please ensure the file exists.")
        exit()

    all_loss_terms = ['LINK', 'ENTROPY', 'RECONSTRUCTION', 'CONTRASTIVE', 'BALANCE', 'REPEL', 'L2']

    # --- Define the combination of loss terms you want to investigate ---
    target_combination = ['LINK', 'ENTROPY', 'BALANCE', 'L2']

    print(f"### Searching for combination: {target_combination} ###")

    # --- Perform a Strict Search ---
    print("\n--- 1. Strict Search (only target terms are active) ---")
    best_strict = find_best_combination(df, target_combination, search_type='strict')

    if best_strict is not None:
        print("Found best strict match:")
        for term in all_loss_terms:
            print(f"  - {term}: {best_strict[term]}")
        print(f"  - Best_E-Score: {best_strict['Best_E-Score']:.4f}")
        print(f"  - Path: {best_strict['Best_Model_Path']}") # Added model path
    else:
        print("No experiment run found where *only* these specific terms were active.")

    # --- Perform a Relaxed Search ---
    print("\n--- 2. Relaxed Search (target terms must be active, others can be) ---")
    best_relaxed = find_best_combination(df, target_combination, search_type='relaxed')

    if best_relaxed is not None:
        print("Found best relaxed match:")
        for term in all_loss_terms:
            print(f"  - {term}: {best_relaxed[term]}")
        print(f"  - Best_E-Score: {best_relaxed['Best_E-Score']:.4f}")
        print(f"  - Path: {best_relaxed['Best_Model_Path']}") # Added model path
    else:
        print(f"No experiment run found where all of {target_combination} were active.")

    # --- For comparison, show the overall best result ---
    print("\n" + "="*60)
    overall_best_run = df.loc[df['Best_E-Score'].idxmax()]
    print("Overall Best-Performing Configuration (for comparison):")
    for term in all_loss_terms:
        print(f"  - {term}: {overall_best_run[term]}")
    print(f"  - Best_E-Score: {overall_best_run['Best_E-Score']:.4f}")
    print(f"  - Path: {overall_best_run['Best_Model_Path']}") # Added model

### Searching for combination: ['LINK', 'ENTROPY', 'BALANCE', 'L2'] ###

--- 1. Strict Search (only target terms are active) ---
No experiment run found where *only* these specific terms were active.

--- 2. Relaxed Search (target terms must be active, others can be) ---
Found best relaxed match:
  - LINK: 100
  - ENTROPY: 0.01
  - RECONSTRUCTION: 1.0
  - CONTRASTIVE: 0.1
  - BALANCE: 0.01
  - REPEL: 0.0
  - L2: 0.01
  - Best_E-Score: 0.8833
  - Path: models/grid_search/STF_HC_Voto_Relatorio/best_model_DiffPool_20250903_135039_lr0.0001_hd_32_bs4_dec0.1_lk100.0_en0.01_rc1.0_ct0.1_bl0.01_rp0.0_l20.01_rep02.pth

Overall Best-Performing Configuration (for comparison):
  - LINK: 0
  - ENTROPY: 0.0
  - RECONSTRUCTION: 0.0
  - CONTRASTIVE: 0.0
  - BALANCE: 1.0
  - REPEL: 0.0
  - L2: 0.0
  - Best_E-Score: 0.8919
  - Path: nan


## Comparing configs


In [5]:
import pandas as pd

df = pd.read_excel("experiment_results_full.xlsx")
df.to_csv("experiment_results_full.csv", index=False)

In [11]:
import scipy.stats as stats

# Extract the scores for cfg_03650 and cfg_00000
# Best_E-Score_median,Best_E-Score_std

scores_cfg_03650 = df.loc[df["Loss_Config_ID"] == "cfg_03650", "F1"]
scores_cfg_00000 = df.loc[df["Loss_Config_ID"] == "cfg_00000", "F1"]

# Step 1: Check normality (Shapiro-Wilk test)
shapiro_cfg_03650 = stats.shapiro(scores_cfg_03650)
shapiro_cfg_00000 = stats.shapiro(scores_cfg_00000)

# Step 2: Check variance homogeneity (Levene’s test)
levene_test = stats.levene(scores_cfg_03650, scores_cfg_00000)

# Step 3: Choose test
# If both normal and variances equal -> independent t-test
# Otherwise -> Mann–Whitney U test

if shapiro_cfg_03650.pvalue > 0.05 and shapiro_cfg_00000.pvalue > 0.05 and levene_test.pvalue > 0.05:
    test_name = "Independent t-test"
    test_stat, p_value = stats.ttest_ind(scores_cfg_03650, scores_cfg_00000, equal_var=True)
else:
    test_name = "Mann–Whitney U test"
    test_stat, p_value = stats.mannwhitneyu(scores_cfg_03650, scores_cfg_00000, alternative="two-sided")

shapiro_cfg_03650, shapiro_cfg_00000, levene_test, (test_name, test_stat, p_value)


(ShapiroResult(statistic=0.9525996391668027, pvalue=0.5808355980860763),
 ShapiroResult(statistic=0.8757920730595077, pvalue=0.25028478049900893),
 LeveneResult(statistic=0.6612146767273297, pvalue=0.4429059366009852),
 ('Independent t-test', 1.7516148772374494, 0.12330215691537309))